# Predictive baselines — MLP uncertainty heads

Mean–Variance NLL, MC-Dropout and Deep Ensembles, all on the shared lakehouse
plan features (`PlanFeatureAdapter` wrapping `TrinoNumericPlanEncoder`). Same
data loading / split / targets as `test_predictive_baseline_tlstm.ipynb`, so
the metrics drop straight into the evaluation workbook.

In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.load import load_aligned_plans_and_runs

from uncertainty_prediction.src import *
from uncertainty_prediction.config import *

from uncertainty_prediction.baselines.predictive.common import (
    build_feature_dataset,
    evaluate_gaussian_predictions,
    print_metric_headers_for_excel,
    print_metrics_for_excel,
)
from uncertainty_prediction.baselines.predictive.mean_variance_nll import MeanVarianceNLL
from uncertainty_prediction.baselines.predictive.mc_dropout import MCDropout
from uncertainty_prediction.baselines.predictive.deep_ensembles import DeepEnsemble

import torch

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
"""
Load and extract query plans and features
"""
# any set of plans is fine here (plans are identical across runs)
queries_dir = "/mnt/lakehouse-raw-results/tpcds/lakehouse-a/20260222-191819Z/queries"

plans_by_query, runs_by_query, common = load_aligned_plans_and_runs(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    min_runs=1,
    min_points_per_run=2,
    require_cols=(XCOL, YCOL),
)

In [ ]:
"""
Split into training and testing sets (same protocol as the TLSTM baseline)
"""
train_qids, test_qids = split_query_ids(common, seed=SEED, test_frac=TEST_FRAC)
print("n_train:", len(train_qids), "n_test:", len(test_qids))

runs_train, runs_test = make_train_test_run_split(
    runs_by_query, train_qids=train_qids, test_qids=test_qids,
)

In [ ]:
"""
Build shared standardised plan features + log-runtime targets
"""
data = build_feature_dataset(
    plans_by_query=plans_by_query,
    runs_by_query=runs_by_query,
    train_qids=train_qids,
    test_qids=test_qids,
    xcol=XCOL,
    runtime_mode="mean",
)

X_train, X_test = data["X_train"], data["X_test"]
y_train_log, y_test_log = data["y_train_log"], data["y_test_log"]
print("feature_dim:", data["feature_dim"], "| X_train:", X_train.shape, "| X_test:", X_test.shape)

## Mean–Variance NLL

In [ ]:
mv = MeanVarianceNLL(in_dim=data["feature_dim"], device=device, seed=42)
mv.fit(X_train, y_train_log, num_epochs=200, lr=1e-3, verbose=True)

pred = mv.predict_gaussian(X_test)
mv_metrics = evaluate_gaussian_predictions(pred["mu_log"], pred["sigma_log"], y_test_log)
print_metric_headers_for_excel(mv_metrics)
print_metrics_for_excel(mv_metrics)

## MC-Dropout

In [ ]:
mc = MCDropout(in_dim=data["feature_dim"], dropout=0.2, num_samples=100, device=device, seed=42)
mc.fit(X_train, y_train_log, num_epochs=200, lr=1e-3, verbose=True)

pred = mc.predict_gaussian(X_test)
mc_metrics = evaluate_gaussian_predictions(pred["mu_log"], pred["sigma_log"], y_test_log)
print_metric_headers_for_excel(mc_metrics)
print_metrics_for_excel(mc_metrics)

## Deep Ensembles

In [ ]:
de = DeepEnsemble(in_dim=data["feature_dim"], num_members=5, device=device, seed=42)
de.fit(X_train, y_train_log, num_epochs=200, lr=1e-3, verbose=True)

pred = de.predict_gaussian(X_test)
de_metrics = evaluate_gaussian_predictions(pred["mu_log"], pred["sigma_log"], y_test_log)
print_metric_headers_for_excel(de_metrics)
print_metrics_for_excel(de_metrics)

## Side-by-side summary

In [ ]:
summary = pd.DataFrame(
    {
        "Mean-Variance NLL": mv_metrics,
        "MC-Dropout": mc_metrics,
        "Deep Ensembles": de_metrics,
    }
).T
cols = ["mae", "rmse", "median_q_error", "crps", "cov@50", "cov@90", "cov@99", "mpiw", "unc_spearman", "unc_pearson"]
summary[cols]